# PFS AG Log Explorer

Interactive exploration of PFS Auto-Guider actor log files.

**Run all cells** to launch the app, or serve it with:
```
panel serve ag_explorer.ipynb --show
```

Controls:
- **Log directory** — type any path; file list refreshes automatically
- **Log file** — newest first, with file sizes shown
- **Design** — filter to a single design window
- **Panels** — toggle individual plot panels on/off
- **Follow / Tail** — stream the file live; set the refresh interval in seconds


In [ ]:
import importlib.util
import sys
from bisect import bisect_right
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path

import panel as pn
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pn.extension('plotly', 'modal', sizing_mode='stretch_width')


In [ ]:
# Import shared helpers (parsers, DataStore, time utils) from ag_common.py
from ag_common import (
    DataStore,
    HST, _night_bounds_from_store,
    parse_line,
)


In [ ]:
# Define notebook directory path for helpers that still reference _HERE
# (backward-compat with earlier cells).
_HERE = Path().resolve()


In [ ]:
_record_cache: dict[str, dict] = {}


def _fmt_size(n: int) -> str:
    if n < 1024:       return f'{n} B'
    if n < 1_048_576:  return f'{n/1024:.1f} KB'
    return f'{n/1_048_576:.1f} MB'


def _parse_log(path: str) -> dict[str, list]:
    """Parse entire file, caching results so re-selecting is instant."""
    if path not in _record_cache:
        state: dict = {}
        records: dict[str, list] = defaultdict(list)
        with open(path, errors='replace') as f:
            for line in f:
                for rtype, rec in parse_line(line.rstrip('\n'), state):
                    records[rtype].append(rec)
        _record_cache[path] = dict(records)
    return _record_cache[path]


def _parse_from_pos(path: str, pos: int, state: dict) -> tuple[dict[str, list], int]:
    """Read from byte offset *pos*, return (new_records, new_pos)."""
    new_records: dict[str, list] = defaultdict(list)
    with open(path, errors='replace') as f:
        f.seek(pos)
        for line in f:
            for rtype, rec in parse_line(line.rstrip('\n'), state):
                new_records[rtype].append(rec)
        new_pos = f.tell()
    return dict(new_records), new_pos


def _make_store(records: dict[str, list], t_start=None, t_end=None) -> DataStore:
    store = DataStore(window_minutes=None)
    for rtype, recs in records.items():
        for rec in recs:
            if (t_start is None or rec.t >= t_start) and \
               (t_end   is None or rec.t <  t_end):
                store.push(rtype, rec)
    return store


def _last_t(records: dict[str, list]):
    last = None
    for recs in records.values():
        if recs and (last is None or recs[-1].t > last):
            last = recs[-1].t
    return last


def _context_arrays(tel_recs: list, store: 'DataStore') -> list:
    """For each tel_axes record return [design_str, visit_str, frame_str]."""
    designs = store.snapshot('design_changes')
    visits  = store.snapshot('visit_changes')
    guides  = store.snapshot('guide')
    d_times = [r.t for r in designs]
    v_times = [r.t for r in visits]
    g_times = [r.t for r in guides]
    out = []
    for r in tel_recs:
        di = bisect_right(d_times, r.t) - 1
        vi = bisect_right(v_times, r.t) - 1
        gi = bisect_right(g_times, r.t) - 1
        out.append([
            f'd…{str(designs[di].design_id)[-8:]}' if di >= 0 else '—',
            f'v{visits[vi].visit_id}'                  if vi >= 0 else '—',
            str(guides[gi].frame_id)                     if gi >= 0 else '—',
        ])
    return out


def _guide_context_arrays(guide_recs: list, store: 'DataStore') -> list:
    """For each guide record return [design_str, visit_str, frame_str]."""
    designs = store.snapshot('design_changes')
    visits  = store.snapshot('visit_changes')
    d_times = [r.t for r in designs]
    v_times = [r.t for r in visits]
    out = []
    for r in guide_recs:
        di = bisect_right(d_times, r.t) - 1
        vi = bisect_right(v_times, r.t) - 1
        out.append([
            f'd…{str(designs[di].design_id)[-8:]}' if di >= 0 else '—',
            f'v{visits[vi].visit_id}'                  if vi >= 0 else '—',
            str(r.frame_id),
        ])
    return out


def _hst(t):
    """Convert timezone-aware datetime to naive HST for Plotly display."""
    return t.astimezone(HST).replace(tzinfo=None)


def _build_design_labels(designs: list) -> list[str]:
    labels = ['All designs']
    for i, r in enumerate(designs):
        t0 = r.t.astimezone(HST).strftime('%H:%M')
        t1_obj = designs[i + 1].t if i + 1 < len(designs) else None
        t1 = t1_obj.astimezone(HST).strftime('%H:%M') if t1_obj else 'end'
        labels.append(f'd…{str(r.design_id)[-8:]}  {t0}–{t1} HST')
    return labels


def _has_data(store: DataStore) -> bool:
    return any(store.snapshot(k) for k in ('guide', 'focus', 'star_stats', 'camera_counts', 'tel_axes'))


In [ ]:
PANEL_NAMES = ['Guide Offsets', 'InR & Scale', 'Focus', 'Camera Counts', 'Cam Detections', 'Star Quality', 'Telescope']

_PANEL_TITLES = {
    'Guide Offsets':  'Guide offsets',
    'InR & Scale':    'Rotation offset & scale',
    'Focus':          'Focus offsets',
    'Star Quality':   'Star quality / seeing proxy',
    'Camera Counts':  'Camera-half detections (count / design median)',
    'Cam Detections': 'Detected counts per camera-half',
    'Telescope':      'Telescope position',
}
_GUIDE_COLORS = ['#1f77b4', '#ff7f0e', '#d62728', '#9467bd']
_Z_COLORS     = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#17becf']
_CAM_COLORS   = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#17becf']
_CAM_LABELS   = ['AG1', 'AG2', 'AG3', 'AG4', 'AG5', 'AG6']


def _event_shapes_and_annotations(store: DataStore) -> tuple[list, list]:
    shapes, annotations = [], []
    for rec in store.snapshot('visit_changes'):
        x = _hst(rec.t).isoformat()
        is_focus = getattr(rec, 'mode', 'unknown') == 'focus'
        v_color = 'seagreen' if is_focus else 'royalblue'
        v_label = f'[F]v{rec.visit_id}' if is_focus else f'v{rec.visit_id}'
        shapes.append(dict(
            type='line', xref='x', yref='paper',
            x0=x, x1=x, y0=0, y1=1,
            line=dict(color=v_color, width=1.4, dash='dot' if is_focus else 'solid'),
            opacity=0.4, layer='below',
        ))
        annotations.append(dict(
            x=x, y=0.02, xref='x', yref='paper',
            text=v_label, textangle=-90,
            showarrow=False, font=dict(size=9, color=v_color),
            xanchor='left', yanchor='bottom',
        ))
    for rec in store.snapshot('design_changes'):
        x = _hst(rec.t).isoformat()
        shapes.append(dict(
            type='line', xref='x', yref='paper',
            x0=x, x1=x, y0=0, y1=1,
            line=dict(color='darkorange', width=1.4, dash='dash'),
            opacity=0.3, layer='below',
        ))
        annotations.append(dict(
            x=x, y=0.02, xref='x', yref='paper',
            text=f'd{str(rec.design_id)[-8:]}', textangle=-90,
            showarrow=False, font=dict(size=9, color='darkorange'),
            xanchor='right', yanchor='bottom',
        ))
    for rec in store.snapshot('reconfigs'):
        x = _hst(rec.t).isoformat()
        label = ' '.join(f'{k}={v}' for k, v in rec.params.items())
        shapes.append(dict(
            type='line', xref='x', yref='paper',
            x0=x, x1=x, y0=0, y1=1,
            line=dict(color='mediumpurple', width=1.0, dash='dot'),
            opacity=0.4, layer='below',
        ))
        annotations.append(dict(
            x=x, y=0.98, xref='x', yref='paper',
            text=label, textangle=-90,
            showarrow=False, font=dict(size=8, color='mediumpurple'),
            xanchor='right', yanchor='top',
        ))
    return shapes, annotations



def _mc_step_traces(reconfigs, x_start, x_end, yaxis, legendgroup):
    """Return ±max_correction step-function traces, or [] if none found."""
    mc = sorted(
        ((r.t, float(r.params['max_correction'])) for r in reconfigs if 'max_correction' in r.params),
        key=lambda p: p[0],
    )
    if not mc:
        return []
    ts = [_hst(t) for t, _ in mc] + [x_end]
    vals = [v for _, v in mc] + [mc[-1][1]]
    traces = []
    for sign in (1, -1):
        traces.append(go.Scatter(
            x=ts, y=[sign * v for v in vals],
            name='max_correction',
            mode='lines',
            line=dict(color='rgba(220,50,50,0.5)', width=1.2, dash='dot'),
            line_shape='hv',
            legendgroup=legendgroup,
            showlegend=False,
            xaxis='x', yaxis=yaxis,
            hovertemplate=f'max_correction: %{{y:.1f}} arcsec<extra></extra>',
        ))
    return traces



# ── Per-camera-half detection series helpers ─────────────────────────────────

_CAM_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']


def _camera_half_series(store: 'DataStore', db_data: dict | None, fid_to_t: dict | None = None) -> dict:
    """Build per-camera-half detection count time series.

    Parameters
    ----------
    store : DataStore
        Current data store (used as fallback when *db_data* is absent).
    db_data : dict | None
        Dict returned by ``_load_db_data``.  If available, the ``'detected'``
        and ``'exposure_info'`` tables are used to compute per-half counts with
        a zero-fill for frames that produced no detections on a given half.
    fid_to_t : dict | None
        Unused (kept for future use).

    Returns
    -------
    dict[str, tuple[list, list]]
        Keys are ``'AG1L'`` … ``'AG6R'`` (DB path, 12 keys) or
        ``'AG1'`` … ``'AG6'`` (log fallback, 6 keys).
        Each value is a ``(times, counts)`` pair where *times* are HST-aware
        datetimes and *counts* are integer detection counts.
    """
    import pandas as pd

    # ── DB path: per-half split ───────────────────────────────────────────────
    if (
        db_data is not None
        and 'detected' in db_data
        and 'exposure_info' in db_data
    ):
        det  = db_data['detected'].copy()   # one row per detected spot
        info = db_data['exposure_info']

        # SourceDetectionFlags.RIGHT = 1 (bit 0) → 0=left half, 1=right half
        # agc_camera_id is 0-based (0–5); labels become AG1–AG6
        det['half'] = (det['flags'].astype(int) & 1)

        # Count spots per (exposure, camera 0-based, half)
        grouped = (
            det.groupby(['agc_exposure_id', 'agc_camera_id', 'half'])
            .size()
            .reset_index(name='count')
        )

        # Restrict to frames visible in the Guide Offsets panel (log-parsed guideErrors).
        # This ensures the heatmap only lights up when the Guide Offsets panel also has data.
        # Fall back to the DB offsets table if no log guide records are available.
        guide_recs = store.snapshot('guide')
        if guide_recs:
            guiding_ids = {int(r.frame_id) for r in guide_recs if r.frame_id is not None}
        elif 'offsets' in db_data and not db_data['offsets'].empty:
            guiding_ids = set(db_data['offsets']['agc_exposure_id'].unique())
        else:
            guiding_ids = None
        if guiding_ids is not None:
            info = info[info['agc_exposure_id'].isin(guiding_ids)]

        # Full grid: guiding exposures × cameras 0-5 × both halves → zeros for dropouts
        all_exp_ids = info['agc_exposure_id'].unique()
        full_index = pd.MultiIndex.from_product(
            [all_exp_ids, list(range(0, 6)), [0, 1]],
            names=['agc_exposure_id', 'agc_camera_id', 'half'],
        )
        full_df = full_index.to_frame(index=False)
        full_df = full_df.merge(grouped, on=['agc_exposure_id', 'agc_camera_id', 'half'], how='left')
        full_df['count'] = full_df['count'].fillna(0).astype(int)

        merged = full_df.merge(info[['agc_exposure_id', 'taken_at']], on='agc_exposure_id', how='left')
        merged['t'] = pd.to_datetime(merged['taken_at'])  # already naive HST in DB

        series: dict = {}
        for cam_id in range(0, 6):
            for half, suffix in ((0, 'L'), (1, 'R')):
                label = f'AG{cam_id + 1}{suffix}'
                sub = merged[(merged.agc_camera_id == cam_id) & (merged.half == half)]
                sub = sub.sort_values('t')
                series[label] = (list(sub['t']), list(sub['count']))
        return series

    # ── Log fallback: per-camera totals (no L/R split) ───────────────────────
    cam_recs = store.snapshot('camera_counts')
    if not cam_recs:
        return {}
    series = {f'AG{i + 1}': ([], []) for i in range(6)}
    for rec in sorted(cam_recs, key=lambda r: r.t):
        t_hst = _hst(rec.t)
        for i, count in enumerate(rec.counts):
            label = f'AG{i + 1}'
            series[label][0].append(t_hst)
            series[label][1].append(int(count) if count is not None else 0)
    return series


def _dropout_frames(series: dict, store: 'DataStore | None' = None, threshold: float = 0.5) -> list:
    """Return sorted HST datetimes for frames where any camera-half drops out.

    A camera-half is considered dropped out when its count falls below
    ``threshold × per_design_median`` for that half (and the median is positive).
    Falls back to session median when ``store`` is not provided.

    Parameters
    ----------
    series : dict
        Output of ``_camera_half_series``.
    store : DataStore, optional
        Used to obtain design-change times for per-design median normalisation.
        When ``None``, the session median is used instead.
    threshold : float
        Fraction of design median below which a frame is flagged.

    Returns
    -------
    list[datetime]
        Sorted list of HST-aware dropout datetimes.
    """
    import bisect
    import numpy as np
    dropout_set: set = set()

    # Build per-design time boundaries as naive HST to match both DB and log series times.
    # DB path produces tz-aware HST timestamps; log path produces naive HST via _hst().
    # Using naive HST throughout avoids offset-naive/offset-aware comparison errors.
    d_times_hst: list = []
    if store is not None:
        d_recs = store.snapshot('design_changes')
        d_times_hst = [r.t.astimezone(HST).replace(tzinfo=None) for r in d_recs]

    def _to_naive_hst(t: object) -> object:
        return t.replace(tzinfo=None) if getattr(t, 'tzinfo', None) is None else t.astimezone(HST).replace(tzinfo=None)

    for _label, (times, counts) in series.items():
        if not times:
            continue
        arr = np.asarray(counts, dtype=float)

        if d_times_hst:
            # Per-design median: assign each frame to its active design segment
            frame_dids = [bisect.bisect_right(d_times_hst, _to_naive_hst(t)) - 1
                          for t in times]
            frame_dids = [max(0, d) for d in frame_dids]
            unique_dids = list(dict.fromkeys(frame_dids))
            cutoffs = np.full(len(arr), np.nan)
            for did in unique_dids:
                mask = np.array([d == did for d in frame_dids])
                seg = arr[mask]
                seg = seg[~np.isnan(seg)]
                med = float(np.nanmedian(seg)) if len(seg) else 0.0
                if med > 0:
                    cutoffs[mask] = threshold * med
                else:
                    cutoffs[mask] = np.nan  # consistently 0 → not a dropout
            for t, c, cut in zip(times, arr, cutoffs):
                if not np.isnan(cut) and c < cut:
                    dropout_set.add(t)
        else:
            # Fallback: session median
            med = float(np.nanmedian(arr))
            if med <= 0:
                continue
            cutoff = threshold * med
            for t, c in zip(times, arr):
                if c < cutoff:
                    dropout_set.add(t)

    return sorted(dropout_set)


# ── Camera-half heatmap ───────────────────────────────────────────────────────

def _build_camera_heatmap_figure(store: 'DataStore', db_data: dict | None = None) -> 'go.Figure':
    """Build a per-camera-half detection count heatmap (time × camera-half).

    Each cell is coloured by ``count / session_median`` for that row:
    red = dropout (0), white = normal (~1), blue = elevated (≥ 2).

    Parameters
    ----------
    store : DataStore
        Current data store.
    db_data : dict | None
        DB data dict from ``_load_db_data``; falls back to log counts if None.

    Returns
    -------
    go.Figure
    """
    import numpy as np

    series = _camera_half_series(store, db_data)
    if not series:
        fig = go.Figure()
        fig.add_annotation(
            text='No camera count data available',
            xref='paper', yref='paper',
            x=0.5, y=0.5, showarrow=False,
        )
        return fig

    # AG1L first → displayed at top; Plotly heatmap y[0] = top row
    labels = sorted(series.keys())

    all_times = sorted({t for lbl in labels for t in series[lbl][0]})
    if not all_times:
        return go.Figure()

    t_to_idx = {t: i for i, t in enumerate(all_times)}
    z = np.full((len(labels), len(all_times)), np.nan)
    for row, lbl in enumerate(labels):
        times, counts = series[lbl]
        for t, c in zip(times, counts):
            z[row, t_to_idx[t]] = c

    # Normalise by per-design median so star-density differences between fields cancel
    import bisect
    d_recs = sorted(store.snapshot('design_changes'), key=lambda r: r.t)
    if d_recs:
        d_times = [_hst(r.t) for r in d_recs]
        d_ids   = [r.design_id for r in d_recs]
        frame_dids = [
            d_ids[max(0, bisect.bisect_right(d_times, t) - 1)]
            for t in all_times
        ]
    else:
        frame_dids = [0] * len(all_times)

    unique_dids = list(dict.fromkeys(frame_dids))
    z_norm = np.full_like(z, np.nan)
    medians = np.zeros(z.shape)
    for row in range(len(labels)):
        for did in unique_dids:
            mask = np.array([d == did for d in frame_dids])
            seg = z[row, mask]
            seg = seg[~np.isnan(seg)]
            med = float(np.nanmedian(seg)) if len(seg) else 0.0
            medians[row, mask] = med
            if med > 0:
                z_norm[row, mask] = z[row, mask] / med
            else:
                # Camera consistently has 0 counts for this design → treat as normal (1.0)
                z_norm[row, mask] = 1.0
    customdata = np.stack([z, medians], axis=-1)

    colorscale = [
        [0.00, 'rgb(180,0,0)'],      # dropout  → red
        [0.50, 'rgb(255,255,255)'],  # normal   → white
        [1.00, 'rgb(0,100,200)'],    # elevated → blue
    ]

    heatmap = go.Heatmap(
        x=all_times,
        y=labels,
        z=z_norm.tolist(),
        zmin=0,
        zmax=2,
        colorscale=colorscale,
        colorbar=dict(title='count/median', thickness=10, len=0.5),
        customdata=customdata.tolist(),
        hovertemplate=(
            '%{y}<br>%{x|%H:%M:%S}<br>'
            'count/median (%{customdata[0]:.0f}/%{customdata[1]:.0f})'
            ' = %{z:.2f}<extra></extra>'
        ),
    )

    fig = go.Figure(heatmap)
    fig.update_layout(
        height=280,
        margin=dict(l=70, r=90, t=30, b=40),
        xaxis=dict(title=''),
        yaxis=dict(title=''),
        title=dict(
            text='Camera-half detection counts (relative to median)',
            font=dict(size=11),
            x=0.01,
        ),
        plot_bgcolor='white',
    )
    return fig


def _build_cam_counts_figure(store: 'DataStore', db_data: dict | None = None) -> 'go.Figure':
    """Thin wrapper — delegates to the camera-half heatmap builder."""
    return _build_camera_heatmap_figure(store, db_data)


def _hex_to_rgb(h: str) -> tuple[float, float, float]:
    """Convert '#rrggbb' hex color to (r, g, b) floats in [0, 1]."""
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4))


def _build_scatter_column(store: 'DataStore', show_acq: bool, clamp_offsets: bool = True) -> 'go.Figure':
    """Four scatter plots in one figure — one per relevant time-series panel.

    Rows (top to bottom):
      1. dAz vs dEl         (Guide Offsets)
      2. dInR vs dScale     (InR & Scale)
      3. dFocus vs counts   (Camera Counts + Star Quality, tall)
      4. Peak ADU vs size   (Star Quality)

    Points are colored by guiding mode (rows 1/2/4) or by camera (row 3).
    """
    from plotly.subplots import make_subplots

    _MODE_COLOR = {
        'autoguide': '#2ca02c',
        'acquire':   '#ff7f0e',
        'converge':  '#9467bd',
        'focus':     'seagreen',
        'unknown':   'gray',
    }

    guide = store.snapshot('guide')
    if not show_acq:
        guide = [r for r in guide if r.mode == 'autoguide']

    ss = store.snapshot('star_stats')
    if not show_acq:
        ss = [r for r in ss if r.mode == 'autoguide']

    cam_counts = store.snapshot('camera_counts')
    if not show_acq:
        cam_counts = [r for r in cam_counts if r.mode == 'autoguide']

    # frame_id -> dfocus lookup for the dFocus vs counts scatter
    fid_to_dfocus = {r.frame_id: r.dfocus for r in guide if r.frame_id is not None}

    fig = make_subplots(
        rows=4, cols=1,
        row_heights=[1, 1, 2, 1],
        subplot_titles=['dAz vs dEl', 'dInR vs dScale', 'dFocus vs counts (per cam)', 'Peak ADU vs PSF size'],
        vertical_spacing=0.07,
    )

    modes = list(dict.fromkeys(r.mode for r in guide))

    # ── Row 1: dAz vs dEl ─────────────────────────────────────────────────────
    for mode in modes:
        for invalid in (False, True):
            pts = [r for r in guide if r.mode == mode and (r.status != 'OK') == invalid]
            if not pts:
                continue
            _col = _MODE_COLOR.get(mode, 'gray')
            fig.add_trace(go.Scatter(
                x=[r.daz for r in pts], y=[r.del_ for r in pts],
                mode='markers', name=mode if not invalid else f'{mode} ✕',
                marker=dict(size=5 if invalid else 4, color=_col, opacity=0.9 if invalid else 0.7,
                            symbol='x' if invalid else 'circle'),
                legendgroup=f'mode_{mode}', showlegend=(not invalid),
                customdata=[r.frame_id for r in pts],
                hovertemplate='dAz: %{x:.4f}<br>dEl: %{y:.4f}<br>frame: %{customdata}<extra></extra>',
            ), row=1, col=1)

    # ── Row 2: dInR vs dScale ─────────────────────────────────────────────────
    for mode in modes:
        for invalid in (False, True):
            pts = [r for r in guide if r.mode == mode and (r.status != 'OK') == invalid]
            if not pts:
                continue
            _col = _MODE_COLOR.get(mode, 'gray')
            fig.add_trace(go.Scatter(
                x=[r.dscale * 1e6 for r in pts], y=[r.dinr for r in pts],
                mode='markers', name=mode if not invalid else f'{mode} ✕',
                marker=dict(size=5 if invalid else 4, color=_col, opacity=0.9 if invalid else 0.7,
                            symbol='x' if invalid else 'circle'),
                legendgroup=f'mode_{mode}', showlegend=False,
                customdata=[r.frame_id for r in pts],
                hovertemplate='dScale: %{x:.2f}×10⁻⁶<br>dInR: %{y:.4f}<br>frame: %{customdata}<extra></extra>',
            ), row=2, col=1)

    # ── Row 3: dFocus vs per-camera counts ────────────────────────────────────
    for cam_idx in range(6):
        pts = [
            (fid_to_dfocus[r.frame_id], r.counts[cam_idx])
            for r in cam_counts
            if r.frame_id in fid_to_dfocus and cam_idx < len(r.counts)
        ]
        if pts:
            dfoc_vals, cnt_vals = zip(*pts)
            fig.add_trace(go.Scatter(
                x=list(dfoc_vals), y=list(cnt_vals),
                mode='markers', name=_CAM_LABELS[cam_idx],
                marker=dict(size=4, color=_CAM_COLORS[cam_idx], opacity=0.7),
                legendgroup=f'cam_{cam_idx}', showlegend=True,
                hovertemplate=(
                    f'{_CAM_LABELS[cam_idx]}<br>'
                    'dFocus: %{x:.3f} mm<br>Counts: %{y}<extra></extra>'
                ),
            ), row=3, col=1)

    # ── Row 4: Peak ADU vs PSF size ───────────────────────────────────────────
    ss_modes = list(dict.fromkeys(r.mode for r in ss))
    for mode in ss_modes:
        pts = [r for r in ss if r.mode == mode]
        fig.add_trace(go.Scatter(
            x=[r.size for r in pts], y=[r.peak for r in pts],
            mode='markers', name=mode,
            marker=dict(size=4, color=_MODE_COLOR.get(mode, 'gray'), opacity=0.7),
            legendgroup=f'mode_{mode}', showlegend=False,
            hovertemplate='Size: %{x:.2f} pix<br>Peak: %{y:.0f} ADU<extra></extra>',
        ), row=4, col=1)

    fig.update_layout(
        margin=dict(l=55, r=10, t=30, b=30),
        showlegend=True,
        legend=dict(orientation='v', x=1.05, y=1, font=dict(size=12)),
        plot_bgcolor='white',
        paper_bgcolor='white',
    )
    _grid = dict(showgrid=True, gridcolor='#eee', zeroline=True, zerolinecolor='#ddd')
    _r1_range = dict(range=[-0.5, 0.5]) if clamp_offsets else {}
    _r2_x_range = dict(range=[-100, 100]) if clamp_offsets else {}  # dScale ×10⁻⁶
    _r2_y_range = dict(range=[-10, 10]) if clamp_offsets else {}    # dInR arcsec
    fig.update_xaxes(title_text='dAz (arcsec)', row=1, col=1, **_grid, **_r1_range, title_font=dict(size=10))
    fig.update_yaxes(title_text='dEl (arcsec)', row=1, col=1, **_grid, **_r1_range, title_font=dict(size=10))
    fig.update_xaxes(title_text='dScale (×10⁻⁶)', row=2, col=1, **_grid, **_r2_x_range, title_font=dict(size=10))
    fig.update_yaxes(title_text='dInR (arcsec)', row=2, col=1, **_grid, **_r2_y_range, title_font=dict(size=10))
    fig.update_xaxes(title_text='dFocus (mm)', row=3, col=1, **_grid, title_font=dict(size=10))
    fig.update_yaxes(title_text='Counts', row=3, col=1, **_grid, title_font=dict(size=10))
    fig.update_xaxes(title_text='PSF size (pix)', row=4, col=1, **_grid, title_font=dict(size=10))
    fig.update_yaxes(title_text='Peak ADU', row=4, col=1, **_grid, title_font=dict(size=10), type='log')

    return fig


def _build_figure(store: DataStore, active_panels: list[str], x_start, x_end, db_data: dict | None = None) -> go.Figure:
    import math
    n = len(active_panels)
    if n == 0:
        return go.Figure()

    GAP = 0.03

    def _dom(i):
        """Y-axis domain [bottom, top] for panel i (0 = topmost panel)."""
        h = (1.0 - (n - 1) * GAP) / n
        return [round(1.0 - (i + 1) * h - i * GAP, 4),
                round(1.0 - i * h - i * GAP, 4)]

    def _yaxes(i):
        """Return (p_ref, s_ref, p_key, s_key) for panel i."""
        p = 2 * i + 1
        s = 2 * i + 2
        p_ref = 'y' if p == 1 else f'y{p}'
        s_ref = f'y{s}'
        p_key = 'yaxis' if p == 1 else f'yaxis{p}'
        s_key = f'yaxis{s}'
        return p_ref, s_ref, p_key, s_key

    fig = go.Figure()
    layout_updates = {}

    # Pre-compute camera-half series (shared across Cam Detections and Camera Counts panels)
    _cam_series = _camera_half_series(store, db_data)

    for i, panel in enumerate(active_panels):
        dom = _dom(i)
        p_ref, s_ref, p_key, s_key = _yaxes(i)
        lg = f'p{i}'

        layout_updates[p_key] = dict(domain=dom)
        layout_updates[s_key] = dict(overlaying=p_ref, side='right')

        if panel == 'Guide Offsets':
            recs = store.snapshot('guide')
            if not show_acq_frames.value:
                recs = [r for r in recs if r.mode == 'autoguide']
            if recs:
                ts  = [_hst(r.t) for r in recs]
                ctx = _guide_context_arrays(recs, store)
                invalid_idx = [j for j, r in enumerate(recs) if r.status != 'OK']
                _azel = guide_coords.value == 'Az/El'
                _pairs = (
                    [('daz', 'dAz'), ('del_', 'dEl')]
                    if _azel else
                    [('dra', 'dRA'), ('ddec', 'dDec')]
                )
                for first, ((attr, label), color) in enumerate(zip(_pairs, _GUIDE_COLORS)):
                    if first == 0:
                        _primary = 'dAz' if _azel else 'dRA'
                        ht = (
                            '<b>%{x|%H:%M:%S} HST</b><br>'
                            f'{_primary}: %{{y:.4f}} arcsec<br>'
                            '──────────────────<br>'
                            'Design: %{customdata[0]}<br>'
                            'Visit:  %{customdata[1]}<br>'
                            'Frame:  %{customdata[2]}'
                            '<extra></extra>'
                        )
                    else:
                        ht = f'{label}: %{{y:.4f}} arcsec<extra></extra>'
                    vals = [getattr(r, attr) for r in recs]
                    fig.add_trace(go.Scatter(
                        x=ts, y=vals,
                        name=label, mode='lines+markers',
                        marker=dict(size=4), line=dict(color=color, width=1.2),
                        legendgroup=lg, xaxis='x', yaxis=p_ref,
                        customdata=ctx, hovertemplate=ht,
                    ))
                    if invalid_idx:
                        fig.add_trace(go.Scatter(
                            x=[ts[j] for j in invalid_idx],
                            y=[vals[j] for j in invalid_idx],
                            name='INVALID_OFFSET' if first == 0 else None,
                            mode='markers',
                            marker=dict(size=8, color=color, symbol='x', line=dict(width=2)),
                            legendgroup=lg, showlegend=(first == 0),
                            xaxis='x', yaxis=p_ref,
                            hovertemplate='%{y:.4f} arcsec — INVALID_OFFSET<extra></extra>',
                        ))
                for tr in _mc_step_traces(store.snapshot('reconfigs'), x_start, x_end, p_ref, lg):
                    fig.add_trace(tr)
            # Dashed threshold lines at ±0.125 and ±0.25 arcsec
            for y_val in (0.125, 0.25, -0.125, -0.25):
                fig.add_shape(
                    type='line', xref='paper', x0=0, x1=1,
                    yref=p_ref, y0=y_val, y1=y_val,
                    line=dict(color='gray', width=0.8, dash='dash'),
                    layer='below',
                )
            upd = dict(title_text='arcsec')
            if not full_offset_range.value:
                upd['range'] = [-0.5, 0.5]
            layout_updates[p_key].update(**upd)


        elif panel == 'InR & Scale':
            recs = store.snapshot('guide')
            if not show_acq_frames.value:
                recs = [r for r in recs if r.mode == 'autoguide']
            if recs:
                ts = [_hst(r.t) for r in recs]
                ctx = _guide_context_arrays(recs, store)
                invalid_idx = [j for j, r in enumerate(recs) if r.status != 'OK']
                dinr_vals = [r.dinr for r in recs]
                fig.add_trace(go.Scatter(
                    x=ts, y=dinr_vals,
                    name='dInR', mode='lines+markers',
                    marker=dict(size=4), line=dict(color='#2ca02c', width=1.2),
                    legendgroup=lg, xaxis='x', yaxis=p_ref,
                    customdata=ctx,
                    hovertemplate='dInR: %{y:.4f} arcsec<extra></extra>',
                ))
                if invalid_idx:
                    fig.add_trace(go.Scatter(
                        x=[ts[j] for j in invalid_idx],
                        y=[dinr_vals[j] for j in invalid_idx],
                        name='INVALID_OFFSET (InR)' if False else None,
                        mode='markers',
                        marker=dict(size=8, color='#2ca02c', symbol='x', line=dict(width=2)),
                        legendgroup=lg, showlegend=False,
                        xaxis='x', yaxis=p_ref,
                        hovertemplate='%{y:.4f} arcsec — INVALID_OFFSET<extra></extra>',
                    ))
                for tr in _mc_step_traces(store.snapshot('reconfigs'), x_start, x_end, p_ref, lg):
                    fig.add_trace(tr)
                fig.add_trace(go.Scatter(
                    x=ts, y=[r.dscale * 1e6 for r in recs],
                    name='dScale ×10⁻⁶', mode='lines+markers',
                    marker=dict(size=4, color='gray'), line=dict(color='gray', width=1.2),
                    legendgroup=lg, xaxis='x', yaxis=s_ref,
                    hovertemplate='dScale: %{y:.4f} ×10⁻⁶<extra></extra>',
                ))
            inr_upd = dict(title_text='dInR (arcsec)',
                           tickfont=dict(color='#2ca02c'),
                           title_font=dict(color='#2ca02c'))
            if not full_offset_range.value:
                inr_upd['range'] = [-10, 10]
            layout_updates[p_key].update(**inr_upd)
            scale_upd = dict(title_text='dScale (×10⁻⁶)',
                                         tickfont=dict(color='gray'),
                                         title_font=dict(color='gray'))
            if not full_offset_range.value:
                scale_upd['range'] = [-100, 100]
            layout_updates[s_key].update(**scale_upd)

        elif panel == 'Focus':
            g = store.snapshot('guide')
            f = store.snapshot('focus')
            if not show_acq_frames.value:
                g = [r for r in g if r.mode == 'autoguide']
                f = [r for r in f if r.mode == 'autoguide']
            if f:
                ts = [_hst(r.t) for r in f]
                for attr, color, label in zip(
                    ['z1', 'z2', 'z3', 'z4', 'z5', 'z6'],
                    _Z_COLORS,
                    ['Z1', 'Z2', 'Z3', 'Z4', 'Z5', 'Z6'],
                ):
                    vals = [getattr(r, attr) for r in f]
                    offline = all(math.isnan(v) for v in vals)
                    fig.add_trace(go.Scatter(
                        x=ts, y=vals,
                        name=f'{label} (offline)' if offline else label,
                        mode='markers',
                        marker=dict(size=4, color='lightgrey' if offline else color),
                        legendgroup=lg, xaxis='x', yaxis=p_ref,
                    ))
            if g:
                ts = [_hst(r.t) for r in g]
                fig.add_trace(go.Scatter(
                    x=ts, y=[r.dfocus for r in g],
                    name='dFocus', mode='lines+markers',
                    marker=dict(size=4, color='black'), line=dict(color='black', width=1.2),
                    legendgroup=lg, xaxis='x', yaxis=p_ref,
                ))
            layout_updates[p_key].update(title_text='Focus offset (mm)')

        elif panel == 'Star Quality':
            ss = store.snapshot('star_stats')
            if not show_acq_frames.value:
                ss = [r for r in ss if r.mode == 'autoguide']
            if ss:
                import pandas as pd
                ts = [_hst(r.t) for r in ss]
                sizes = [r.size for r in ss]
                peaks = [r.peak for r in ss]
                win = max(5, len(ts) // 10)
                s_sizes = pd.Series(sizes)
                s_peaks = pd.Series(peaks)
                roll_size = s_sizes.rolling(win, center=True, min_periods=1).median().tolist()
                err_size = s_sizes.rolling(win, center=True, min_periods=1).std().tolist()
                fig.add_trace(go.Scatter(
                    x=ts, y=peaks,
                    name='peak ADU', mode='lines+markers',
                    marker=dict(size=4, color='#ff7f0e'),
                    line=dict(color='#ff7f0e', width=1),
                    legendgroup=lg, xaxis='x', yaxis=s_ref,
                    zorder=1,
                ))
                fig.add_trace(go.Scatter(
                    x=ts, y=sizes,
                    name='PSF size (px)', mode='markers',
                    marker=dict(size=4, color='#1f77b4', opacity=0.25),
                    error_y=dict(type='data', array=err_size, visible=True,
                                 color='#1f77b4', thickness=1, width=0),
                    legendgroup=lg, xaxis='x', yaxis=p_ref,
                    zorder=2,
                ))
                fig.add_trace(go.Scatter(
                    x=ts, y=roll_size,
                    name='rolling median PSF size', mode='lines', showlegend=False,
                    line=dict(color='#1f77b4', width=3),
                    legendgroup=lg, xaxis='x', yaxis=p_ref,
                    zorder=3,
                ))
            layout_updates[p_key].update(title_text='PSF size (px)',
                                         tickfont=dict(color='#1f77b4'),
                                         title_font=dict(color='#1f77b4'))
            layout_updates[s_key].update(title_text='peak ADU (log)', type='log',
                                         tickfont=dict(color='#ff7f0e'),
                                         title_font=dict(color='#ff7f0e'))

        elif panel == 'Cam Detections':
            if _cam_series:
                has_halves = any(k[-1] in ('L', 'R') for k in _cam_series)
                for label, (times, counts) in sorted(_cam_series.items()):
                    cam_idx = int(label[2]) - 1
                    dash = 'solid' if (not has_halves or label.endswith('L')) else 'dash'
                    fig.add_trace(go.Scatter(
                        x=times, y=counts,
                        name=label,
                        mode='lines+markers',
                        marker=dict(size=4, color=_CAM_COLORS[cam_idx]),
                        line=dict(color=_CAM_COLORS[cam_idx], dash=dash, width=1),
                        legendgroup=label[:3],
                        showlegend=True,
                        xaxis='x', yaxis=p_ref,
                    ))
                title = 'counts' if has_halves else 'counts (log only — fetch DB for L/R split)'
                layout_updates[p_key] = dict(domain=dom, title_text=title, rangemode='tozero')
            else:
                fig.add_annotation(
                    text='No data — click "Fetch DB data" to load camera counts',
                    xref='paper', yref='paper', x=0.5, y=(dom[0] + dom[1]) / 2,
                    showarrow=False, font=dict(size=11, color='gray'), xanchor='center',
                )
                layout_updates[p_key] = dict(domain=dom, title_text='counts', rangemode='tozero')

        elif panel == 'Camera Counts':
            if _cam_series:
                import numpy as np
                labels = sorted(_cam_series.keys())  # AG1L first → top of y-axis
                all_t = sorted({t for lbl in labels for t in _cam_series[lbl][0]})
                if all_t:
                    t_to_k = {t: k for k, t in enumerate(all_t)}
                    z = np.full((len(labels), len(all_t)), np.nan)
                    for row, lbl in enumerate(labels):
                        for t, c in zip(*_cam_series[lbl]):
                            z[row, t_to_k[t]] = c
                    # Assign each frame to the design active at that time
                    import bisect
                    d_recs = sorted(store.snapshot('design_changes'), key=lambda r: r.t)
                    if d_recs:
                        d_times = [_hst(r.t) for r in d_recs]
                        d_ids   = [r.design_id for r in d_recs]
                        frame_dids = [
                            d_ids[max(0, bisect.bisect_right(d_times, t) - 1)]
                            for t in all_t
                        ]
                    else:
                        frame_dids = [0] * len(all_t)

                    # Per-design median normalisation
                    unique_dids = list(dict.fromkeys(frame_dids))
                    z_norm = np.full_like(z, np.nan)
                    medians = np.zeros(z.shape)
                    for row in range(len(labels)):
                        for did in unique_dids:
                            mask = np.array([d == did for d in frame_dids])
                            seg = z[row, mask]
                            seg = seg[~np.isnan(seg)]
                            med = float(np.nanmedian(seg)) if len(seg) else 0.0
                            medians[row, mask] = med
                            if med > 0:
                                z_norm[row, mask] = z[row, mask] / med
                            else:
                                # Camera consistently has 0 counts for this design → treat as normal (1.0)
                                z_norm[row, mask] = 1.0
                    colorscale = [
                        [0.00, 'rgb(180,0,0)'],
                        [0.50, 'rgb(255,255,255)'],
                        [1.00, 'rgb(0,100,200)'],
                    ]
                    for row, lbl in enumerate(labels):
                        cam_times, cam_counts = _cam_series[lbl]
                        # z_norm and raw counts for this camera-half, aligned to all_t
                        z_row = z_norm[row, :].tolist()
                        raw_row = z[row, :].tolist()
                        med_row = medians[row, :].tolist()
                        hover = [
                            f'{lbl}  {t.strftime("%H:%M:%S") if hasattr(t, "strftime") else t}<br>'
                            f'count/median ({r:.0f}/{m:.0f}) = {zv:.2f}'
                            for t, r, m, zv in zip(all_t, raw_row, med_row, z_row)
                        ]
                        fig.add_trace(go.Scatter(
                            x=all_t,
                            y=[lbl] * len(all_t),
                            mode='markers',
                            marker=dict(
                                color=z_row,
                                colorscale=colorscale,
                                cmin=0, cmax=2,
                                size=10,
                                symbol='square',
                                showscale=False,
                            ),
                            text=hover,
                            hoverinfo='text',
                            showlegend=False,
                            name=lbl,
                            xaxis='x', yaxis=p_ref,
                        ))
                    # Grey border + separator lines between each camera pair
                    # range(-1, len+1, 2) → -0.5 (top), 1.5, 3.5, …, 9.5, 11.5 (bottom)
                    for sep in range(-1, len(labels) + 1, 2):
                        fig.add_shape(
                            type='line',
                            x0=0, x1=1, xref='x domain',
                            y0=sep + 0.5, y1=sep + 0.5, yref=p_ref,
                            line=dict(color='rgba(80,80,80,0.5)', width=1.5),
                        )
                layout_updates[p_key] = dict(
                    domain=dom, type='category',
                    categoryorder='array', categoryarray=labels,
                )
                layout_updates[s_key] = dict(visible=False)
            else:
                fig.add_annotation(
                    text='No data — click "Fetch DB data" to load camera counts',
                    xref='paper', yref='paper', x=0.5, y=(dom[0] + dom[1]) / 2,
                    showarrow=False, font=dict(size=11, color='gray'), xanchor='center',
                )
                layout_updates[p_key] = dict(domain=dom, type='category', autorange=True)
                layout_updates[s_key] = dict(visible=False)

        elif panel == 'Telescope':
            axes_recs = store.snapshot('tel_axes')
            state_recs = store.snapshot('tel_state')
            # Primary axis: Az, El (low-cadence Gen2 status)
            if axes_recs:
                ts_ax = [_hst(r.t) for r in axes_recs]
                for attr, label, color in [
                    ('az',  'Az (deg)',  '#1f77b4'),
                    ('el',  'El (deg)',  '#ff7f0e'),
                ]:
                    fig.add_trace(go.Scatter(
                        x=ts_ax, y=[getattr(r, attr) for r in axes_recs],
                        name=label, mode='lines+markers',
                        marker=dict(size=3), line=dict(color=color, width=1.2),
                        legendgroup=lg, xaxis='x', yaxis=p_ref,
                    ))
            # Secondary axis: rot (Gen2, low-cadence), InR, ADC, M2 (per-frame)
            if axes_recs:
                fig.add_trace(go.Scatter(
                    x=ts_ax, y=[r.rot for r in axes_recs],
                    name='Rot/PA (deg)', mode='lines+markers',
                    marker=dict(size=3), line=dict(color='#9467bd', width=1.2),
                    legendgroup=lg, xaxis='x', yaxis=s_ref,
                    hovertemplate='Rot/PA: %{y:.3f} deg<extra></extra>',
                ))
            if state_recs:
                ts_st = [_hst(r.t) for r in state_recs]
                for attr, label, color, dash in [
                    ('adc',    'ADC (deg)',    '#d62728', 'dash'),
                    ('m2_pos3','M2 pos3 (mm)', '#8c564b', 'dot'),
                ]:
                    fig.add_trace(go.Scatter(
                        x=ts_st, y=[getattr(r, attr) for r in state_recs],
                        name=label, mode='lines+markers',
                        marker=dict(size=3), line=dict(color=color, width=1.2, dash=dash),
                        legendgroup=lg, xaxis='x', yaxis=s_ref,
                        hovertemplate=f'{label}: %{{y:.4f}}<extra></extra>',
                    ))
            rot_recs = store.snapshot('tel_rot')
            if rot_recs:
                ts_rot = [_hst(r.t) for r in rot_recs]
                fig.add_trace(go.Scatter(
                    x=ts_rot, y=[r.insrot for r in rot_recs],
                    name='InR (deg)', mode='lines',
                    line=dict(color='#2ca02c', width=1.5),
                    legendgroup=lg, xaxis='x', yaxis=s_ref,
                    hovertemplate='InR: %{y:.4f} deg<extra></extra>',
                ))
            layout_updates[p_key].update(title_text='Az / El (deg)')
            layout_updates[s_key].update(title_text='Rot / InR / ADC (deg) · M2 (mm)')

    # ── Align zeros on dual y-axis panels ─────────────────────────────────────
    for i, panel in enumerate(active_panels):
        p_ref, s_ref, p_key, s_key = _yaxes(i)
        if layout_updates.get(s_key, {}).get('type') == 'log':
            continue  # log scale has no zero to align
        # Don't override an explicit range (e.g. clamped InR ±10 arcsec or dScale ±100)
        if 'range' in layout_updates.get(p_key, {}) or 'range' in layout_updates.get(s_key, {}):
            continue
        p_traces = [t for t in fig.data if getattr(t, 'yaxis', None) == p_ref]
        s_traces = [t for t in fig.data if getattr(t, 'yaxis', None) == s_ref]
        if not p_traces or not s_traces:
            continue
        def _finite(traces):
            return [float(v) for t in traces for v in (t.y or [])
                    if v is not None and not math.isnan(float(v))]
        pv, sv = _finite(p_traces), _finite(s_traces)
        if not pv or not sv:
            continue
        lo_p, hi_p = min(pv), max(pv)
        lo_s, hi_s = min(sv), max(sv)
        # 5 % padding
        pad_p = max(abs(hi_p - lo_p) * 0.05, 1e-9)
        pad_s = max(abs(hi_s - lo_s) * 0.05, 1e-9)
        lo_p -= pad_p; hi_p += pad_p
        lo_s -= pad_s; hi_s += pad_s
        if lo_p >= 0 or hi_p <= 0 or lo_s >= 0 or hi_s <= 0:
            continue  # zero outside a range — nothing to align
        # Pick the fraction f in [0,1] where zero must sit so both datasets fit
        f_p = -lo_p / (hi_p - lo_p)
        f_s = -lo_s / (hi_s - lo_s)
        f = max(f_p, f_s)
        r = f / (1.0 - f)  # ratio |lo| / hi at aligned zero
        def _align(lo, hi):
            if r * hi >= -lo:      # negative side is the constraint
                return -r * hi, hi
            else:                   # positive side is the constraint
                return lo, -lo / r
        layout_updates[p_key]['range'] = list(_align(lo_p, hi_p))
        layout_updates[s_key]['range'] = list(_align(lo_s, hi_s))

    shapes, event_annotations = _event_shapes_and_annotations(store)
    shapes = shapes

    title_annotations = [
        dict(
            x=0, y=_dom(i)[1],
            xref='paper', yref='paper',
            text=f'<b>{_PANEL_TITLES[p]}</b>',
            xanchor='left', yanchor='bottom',
            showarrow=False, font=dict(size=11),
        )
        for i, p in enumerate(active_panels)
    ]

    fig.update_layout(
        **layout_updates,
        xaxis=dict(
            domain=[0, 1],
            range=[_hst(x_start).isoformat(), _hst(x_end).isoformat()],
            title_text='Time (HST)',
            showspikes=True,
            spikemode='across',
            spikesnap='cursor',
            spikecolor='rgba(100,100,100,0.5)',
            spikethickness=1,
            spikedash='solid',
        ),
        height=max(300, 260 * n),
        hovermode='x unified',
        legend=dict(orientation='v', x=-0.12, y=1, xanchor='right', tracegroupgap=12),
        margin=dict(l=180, r=30, t=40, b=60),
        shapes=shapes,
        annotations=event_annotations + title_annotations,
    )
    return fig


In [ ]:
# ── Widgets ───────────────────────────────────────────────────────────────────

_LOGS_DEFAULT = str(_HERE / 'logs')

dir_input = pn.widgets.TextInput(
    name='Log directory', value=_LOGS_DEFAULT, width=280,
)
file_select = pn.widgets.Select(name='Log file', width=280)
show_all_files = pn.widgets.Checkbox(name='Show all files (incl. <1 MB)', value=False)
show_acq_frames = pn.widgets.Checkbox(name='Show non-science frames (acquire/converge/focus)', value=False)
full_offset_range = pn.widgets.Checkbox(name='Guide offsets: full y-range', value=False)
guide_coords = pn.widgets.RadioButtonGroup(
    name='Guide offset coords', options=['Az/El', 'RA/Dec'], value='Az/El',
    button_type='default', width=200,
)
design_select = pn.widgets.Select(name='Design', width=280)
panels_check = pn.widgets.CheckBoxGroup(
    name='Panels', options=PANEL_NAMES, value=PANEL_NAMES,
)
follow_toggle = pn.widgets.Toggle(
    name='▶ Follow / Tail', button_type='success', width=150,
)
interval_input = pn.widgets.IntInput(
    name='Refresh (s)', value=5, start=1, end=300, step=1, width=100,
)
tail_window_select = pn.widgets.Select(
    name='Tail window',
    options={
        'All data': 'all',
        'Last 5 min': '5m',
        'Last 10 min': '10m',
        'Last 30 min': '30m',
        'Current visit': 'visit',
        'Current design': 'design',
    },
    value='all',
    width=150,
)
status_md = pn.pane.Markdown('', width=280)
plot_pane = pn.pane.Plotly(go.Figure(), sizing_mode='stretch_width', min_height=400)
scatter_col_pane = pn.pane.Plotly(go.Figure(), sizing_mode='stretch_height', width=650)
cam_counts_pane = pn.pane.Plotly(go.Figure(), sizing_mode='stretch_width', min_height=520)
db_status_md = pn.pane.Markdown('', width=280, margin=(2, 5))
design_info_md = pn.pane.Markdown('', width=280, margin=(0, 5))
fetch_db_btn = pn.widgets.Button(
    name='⬇ Fetch DB data', button_type='primary', width=200,
)

# ── Internal state ────────────────────────────────────────────────────────────

_record_cache_local: dict[str, dict] = {}
_busy = [False]
_follow_cb = [None]
_tail_state: dict = {}
_tail_pos = [0]
_tail_records: dict[str, list] = {}
_current_store: list = [None]        # [DataStore | None] — store for zoom sync
_current_db_data: list = [None]      # [dict | None]      — DB data for zoom sync
_current_x_range: list = [None, None] # [x_start, x_end]  — for maximize re-render
_cam_counts_base_fig: list = [None]  # [go.Figure | None] — full-range heatmap base
_sync_timer: list = [None]           # [threading.Timer | None] — debounce handle


# ── Helpers ───────────────────────────────────────────────────────────────────

def _log_path():
    d = Path(dir_input.value.strip())
    v = file_select.value
    return d / v if v else None


def _refresh_file_list(directory: str):
    p = Path(directory.strip())
    if p.is_dir():
        all_paths = sorted(p.glob('*.log'), reverse=True)
        paths = all_paths if show_all_files.value else [
            pp for pp in all_paths if pp.stat().st_size >= 1_048_576
        ]
    else:
        paths = []
    opts = {f'{pp.name}  ({_fmt_size(pp.stat().st_size)})': pp.name for pp in paths}
    file_select.param.update(options=opts, value=list(opts.values())[0] if opts else None)


def _refresh_design_list(records: dict):
    designs = records.get('design_changes', [])
    design_select.param.update(
        options=_build_design_labels(designs),
        value='All designs',
    )


def _current_records():
    """Return records to plot: tail buffer when following, cache otherwise."""
    if follow_toggle.value and _tail_records:
        return _tail_records
    p = _log_path()
    return _parse_log(str(p)) if p and p.exists() else {}


def _tail_window_bounds(records: dict):
    """Return (t_start, t_end) for the active tail window, or (None, None) for all data."""
    window = tail_window_select.value
    if window == '5m':
        return datetime.now(tz=HST) - timedelta(minutes=5), None
    if window == '10m':
        return datetime.now(tz=HST) - timedelta(minutes=10), None
    if window == '30m':
        return datetime.now(tz=HST) - timedelta(minutes=30), None
    if window == 'visit':
        visits = records.get('visit_changes', [])
        return (visits[-1].t, None) if visits else (None, None)
    if window == 'design':
        designs = records.get('design_changes', [])
        return (designs[-1].t, None) if designs else (None, None)
    return None, None  # 'all'


_frame_ts: dict[int, object] = {}  # frame_id -> datetime, updated each render


def _on_scatter_click(event):
    """Draw a vertical marker on the time-series at the clicked frame."""
    cd = event.new
    if not cd or 'points' not in cd or not cd['points']:
        return
    frame_id = cd['points'][0].get('customdata')
    if frame_id is None or frame_id not in _frame_ts:
        return
    ts = _hst(_frame_ts[frame_id]).isoformat()
    fig = plot_pane.object
    if fig is None:
        return
    shapes = [s for s in (fig.layout.shapes or []) if getattr(s, 'name', None) != 'click_marker']
    shapes = list(shapes) + [dict(
        name='click_marker',
        type='line', xref='x', yref='paper',
        x0=ts, x1=ts, y0=0, y1=1,
        line=dict(color='crimson', width=2, dash='dash'),
        opacity=0.8, layer='above',
    )]
    fig.update_layout(shapes=shapes)
    plot_pane.param.trigger('object')


scatter_col_pane.param.watch(_on_scatter_click, 'click_data')


# ── Cross-pane zoom synchronisation ──────────────────────────────────────────

def _parse_plotly_ts(s: str):
    """Parse a Plotly x-axis timestamp string (HST, no tz info) to a UTC datetime."""
    from datetime import timezone
    return datetime.fromisoformat(s.split('.')[0]).replace(tzinfo=HST).astimezone(timezone.utc)


def _sync_panes(t0_str, t1_str) -> None:
    """Rebuild scatter and shift heatmap viewport after a main-figure zoom.

    Parameters
    ----------
    t0_str, t1_str : str | None
        HST timestamp strings from Plotly relayout_data.  Both None means
        autorange reset.
    """
    store = _current_store[0]
    if store is None:
        return

    if t0_str and t1_str:
        try:
            t0_utc = _parse_plotly_ts(t0_str)
            t1_utc = _parse_plotly_ts(t1_str)
        except (ValueError, TypeError):
            return
        sliced = store.slice(t0_utc, t1_utc)
    else:
        sliced = store

    # Rebuild scatter column with only the visible data points
    scatter_col_pane.object = _build_scatter_column(
        sliced, show_acq_frames.value, clamp_offsets=not full_offset_range.value,
    )



def _on_main_relayout(event) -> None:
    """Debounced callback: syncs scatter + heatmap when the main figure is zoomed/panned."""
    import threading
    rd = event.new
    if not isinstance(rd, dict):
        return
    if _sync_timer[0] is not None:
        _sync_timer[0].cancel()

    def _execute():
        if rd.get('xaxis.autorange') is True:
            pn.state.execute(lambda: _sync_panes(None, None))
        elif 'xaxis.range[0]' in rd and 'xaxis.range[1]' in rd:
            x0, x1 = rd['xaxis.range[0]'], rd['xaxis.range[1]']
            pn.state.execute(lambda: _sync_panes(x0, x1))

    _sync_timer[0] = threading.Timer(0.25, _execute)
    _sync_timer[0].start()


plot_pane.param.watch(_on_main_relayout, 'relayout_data')


def _load_db_data(design_ids: list[int], data_dir: str, log_path: str = '') -> tuple[dict, str]:
    """Load DB CSV files downloaded by ``query-ag-data --save`` for each design_id.

    Returns a tuple of:
    - dict with keys 'detected', 'matched', 'guide_catalog' (concatenated DataFrames)
    - status markdown string summarising what was loaded and any errors
    """
    import pandas as pd
    from pathlib import Path

    tables = {'exposure_info': [], 'detected': [], 'matched': [], 'offsets': [], 'guide_catalog': []}
    errors: list[str] = []
    missing_designs: list[int] = []
    p = Path(data_dir.strip())

    if not p.is_dir():
        msg = f'⚠️ **DB data:** directory not found: `{data_dir}`'
        return {k: pd.DataFrame() for k in tables}, msg

    for design_id in design_ids:
        found_any = False
        for tname in tables:
            fname = p / f'{design_id}-{tname}.csv'
            if fname.exists():
                found_any = True
                try:
                    tables[tname].append(pd.read_csv(fname))
                except Exception as exc:
                    errors.append(f'`{fname.name}`: {exc}')
        if not found_any:
            missing_designs.append(design_id)

    result = {k: pd.concat(v, ignore_index=True) if v else pd.DataFrame() for k, v in tables.items()}

    parts: list[str] = []
    total = sum(len(df) for df in result.values() if not df.empty)
    if total == 0 and not errors:
        log_name = Path(log_path).name if log_path else '<logfile>'
        hint = f'Run: `query-ag-data --log {log_name} --save`'
        parts.append(f'ℹ️ **DB data:** no CSV files found in `{p.name}/`. {hint}')
    else:
        counts = '  ·  '.join(
            f'**{k}:** {len(df):,}' for k, df in result.items() if not df.empty
        ) or 'no data'
        parts.append(f'📊 **DB data loaded —** {counts}')
        if missing_designs:
            parts.append(f'⚠️ No CSV files for designs: {missing_designs}')
        if errors:
            parts.append('❌ Errors: ' + '; '.join(errors))

    return result, '\n'.join(parts)


_fetch_confirm_pending = [False]
_fetch_confirm_timer: list = [None]


def _reset_fetch_btn():
    """Restore button to its default state and clear confirmation flag."""
    _fetch_confirm_pending[0] = False
    fetch_db_btn.param.update(name='⬇ Fetch DB data', button_type='primary', disabled=False)


def _on_fetch_db(_event=None):
    import threading
    from pathlib import Path

    p = _log_path()
    if not p or not p.exists():
        db_status_md.object = "No log file selected."
        db_status_md.visible = True
        return

    records = _current_records()
    design_ids = [r.design_id for r in records.get("design_changes", [])]
    if not design_ids:
        db_status_md.object = "No design IDs found in log."
        db_status_md.visible = True
        return

    out_dir = p.parent

    # Check if CSV files already exist for any design
    existing = [did for did in design_ids if (out_dir / f"{did}-detected.csv").exists()]

    if existing and not _fetch_confirm_pending[0]:
        # First click: data exists — ask for confirmation via button state
        _fetch_confirm_pending[0] = True
        n_exist = len(existing)
        fetch_db_btn.param.update(
            name=f'⚠ {n_exist} design(s) cached — click again to re-fetch',
            button_type='warning',
        )
        db_status_md.object = f'ℹ️ CSV files already exist for {n_exist} design(s). Click the button again to overwrite them, or wait 5 s to cancel.'
        db_status_md.visible = True
        # Auto-reset after 5 s if user doesn't confirm
        if _fetch_confirm_timer[0]:
            _fetch_confirm_timer[0].cancel()
        t = threading.Timer(5.0, lambda: pn.state.execute(_reset_fetch_btn))
        _fetch_confirm_timer[0] = t
        t.daemon = True
        t.start()
        return

    # Either no existing files, or this is the confirmation click — proceed
    if _fetch_confirm_timer[0]:
        _fetch_confirm_timer[0].cancel()
        _fetch_confirm_timer[0] = None
    _fetch_confirm_pending[0] = False

    # Show immediate feedback before starting thread
    n = len(design_ids)
    db_status_md.object = f"Fetching DB data for {n} design(s)..."
    db_status_md.visible = True
    fetch_db_btn.param.update(name="Fetching...", disabled=True, button_type='primary')

    out_dir = p.parent

    def _run():
        saved: list[str] = []
        errors: list[str] = []

        try:
            from agActor.utils.diagnostics import query_ag_data
        except ImportError as exc:
            pn.state.execute(lambda: [
                fetch_db_btn.param.update(name="Fetch DB data", disabled=False),
                setattr(db_status_md, "object", f"Import error: {exc}"),
                setattr(db_status_md, "visible", True),
            ])
            return

        for i, did in enumerate(design_ids):
            pn.state.execute(lambda i=i: fetch_db_btn.param.update(
                name=f"Fetching {i+1}/{len(design_ids)}...", disabled=True,
            ))
            try:
                tables = query_ag_data(design_id=did)
                for tname, df in tables.items():
                    path = out_dir / f"{did}-{tname}.csv"
                    df.to_csv(path, index=False)
                    saved.append(path.name)
            except Exception as exc:
                errors.append(f"design {did}: {exc}")

        def _finish():
            fetch_db_btn.param.update(name="Fetch DB data", disabled=False)
            if not saved and not errors:
                msg = "No data returned (no frames for these designs?)."
            elif errors and not saved:
                msg = "Errors: " + " | ".join(errors)
            elif errors:
                msg = f"Saved {len(saved)} file(s).  Errors: " + " | ".join(errors)
            else:
                msg = f"Saved {len(saved)} file(s) to {out_dir}"
            db_status_md.object = msg
            db_status_md.visible = True
            if saved:
                _render()

        pn.state.execute(_finish)

    threading.Thread(target=_run, daemon=True).start()

fetch_db_btn.on_click(_on_fetch_db)


def _render():
    p = _log_path()
    if not p:
        plot_pane.object = go.Figure()
        scatter_col_pane.object = go.Figure()
        status_md.object = '⚠️ No log file selected.'
        return
    try:
        records = _current_records()
        designs = records.get('design_changes', [])
        labels  = list(design_select.options) or ['All designs']
        sel     = design_select.value or 'All designs'

        if follow_toggle.value and tail_window_select.value != 'all':
            t_start, t_end = _tail_window_bounds(records)
            if t_start is not None:
                store = _make_store(records, t_start, t_end)
                x_start = t_start
                x_end = t_end or _last_t(records) or t_start
            else:
                store = _make_store(records)
                x_start, x_end = _night_bounds_from_store(store)
        elif sel == 'All designs':
            store = _make_store(records)
            x_start, x_end = _night_bounds_from_store(store)
        else:
            d_idx   = max(0, labels.index(sel) - 1)
            t_start = designs[d_idx].t
            t_end   = designs[d_idx + 1].t if d_idx + 1 < len(designs) else None
            store   = _make_store(records, t_start, t_end)
            x_start = t_start
            x_end   = t_end or _last_t(records) or t_start

        if not _has_data(store):
            plot_pane.object = go.Figure()
            scatter_col_pane.object = go.Figure()
            total = sum(len(v) for v in records.values())
            if total == 0:
                status_md.object = f'⚠️ **{p.name}** — empty or unrecognised format.'
            else:
                status_md.object = (
                    f'⚠️ **{p.name}** — {total} lines parsed, no AG guiding data.'
                )
            return

        _current_store[0] = store

        # Load DB data when any panel that uses it is active
        need_db = any(p in panels_check.value for p in ('Camera Counts', 'Cam Detections'))
        if need_db:
            design_ids = [r.design_id for r in store.snapshot('design_changes')]
            db_data, db_status = _load_db_data(
                design_ids, str(Path(_log_path()).parent), log_path=str(_log_path()),
            )
            db_status_md.object = db_status
            db_status_md.visible = True
        else:
            db_data = _current_db_data[0]
            db_status_md.visible = False
        _current_db_data[0] = db_data

        active_panels = list(panels_check.value)
        fig = _build_figure(store, active_panels, x_start, x_end, db_data=db_data)
        plot_pane.object = fig
        _current_x_range[0] = x_start
        _current_x_range[1] = x_end
        _frame_ts.clear()
        _frame_ts.update({r.frame_id: r.t for r in store.snapshot('guide') if r.frame_id is not None})
        scatter_col_pane.object = _build_scatter_column(store, show_acq_frames.value, clamp_offsets=not full_offset_range.value)

        cam_counts_pane.visible = False  # Camera Counts renders inside the main figure
        g = len(store.snapshot('guide'))
        v = len(store.snapshot('visit_changes'))
        d = len(store.snapshot('design_changes'))
        status_md.object = f'**Exposures:** {g} · **Visits:** {v} · **Designs:** {d}'

    except Exception as exc:
        import traceback
        plot_pane.object = go.Figure()
        scatter_col_pane.object = go.Figure()
        status_md.object = f'❌ `{type(exc).__name__}: {exc}`'
        traceback.print_exc()


# ── Tail mode ─────────────────────────────────────────────────────────────────

def _stop_follow():
    if _follow_cb[0] is not None:
        try:
            _follow_cb[0].stop()
        except Exception:
            pass
        _follow_cb[0] = None


def _init_tail():
    global _tail_records, _tail_state
    p = _log_path()
    if not p or not p.exists():
        return
    records = _parse_log(str(p))
    _tail_records = {k: list(v) for k, v in records.items()}
    _tail_state = {}
    with open(str(p), errors='replace') as f:
        f.seek(0, 2)
        _tail_pos[0] = f.tell()


def _tail_tick():
    p = _log_path()
    if not p or not p.exists():
        return
    try:
        new_recs, new_pos = _parse_from_pos(str(p), _tail_pos[0], _tail_state)
        _tail_pos[0] = new_pos
        if new_recs:
            for rtype, recs in new_recs.items():
                _tail_records.setdefault(rtype, []).extend(recs)
            if 'design_changes' in new_recs:
                _busy[0] = True
                design_select.options = _build_design_labels(
                    _tail_records.get('design_changes', [])
                )
                _busy[0] = False
            _render()
    except Exception as exc:
        status_md.object = f'❌ Tail error: `{exc}`'


# ── Widget callbacks ──────────────────────────────────────────────────────────

def _load_file():
    """Parse the selected file, refresh the design dropdown, and render.
    Called explicitly so directory changes always trigger a full reload even
    when file_select.value hasn't changed."""
    v = file_select.value
    if not v:
        plot_pane.object = go.Figure()
        status_md.object = '⚠️ No log file selected.'
        return
    p = Path(dir_input.value.strip()) / v
    if not p.exists():
        status_md.object = f'⚠️ Not found: `{p}`'
        return
    _busy[0] = True
    records = _parse_log(str(p))
    _refresh_design_list(records)
    _busy[0] = False
    _render()


@pn.depends(dir_input.param.value, watch=True)
def _on_dir(value):
    _stop_follow()
    follow_toggle.value = False
    _busy[0] = True          # suppress _on_file watcher during list refresh
    _refresh_file_list(value)
    _busy[0] = False
    _load_file()             # explicit reload regardless of whether value changed


@pn.depends(file_select.param.value, watch=True)
def _on_file(value):
    if _busy[0]:
        return               # _on_dir is driving; it will call _load_file()
    _stop_follow()
    follow_toggle.value = False
    _load_file()


def _update_design_info(sel_value: str):
    """Populate design_info_md with full design ID and visit list."""
    if not sel_value or sel_value == 'All designs':
        design_info_md.object = ''
        return
    records = _current_records()
    designs = records.get('design_changes', [])
    visits  = records.get('visit_changes', [])
    suffix = sel_value.split('d\u2026')[-1][:8]
    matched = [r for r in designs if str(r.design_id).endswith(suffix)]
    if not matched:
        design_info_md.object = ''
        return
    dr  = matched[0]
    idx = designs.index(dr)
    t_start = dr.t
    t_end   = designs[idx + 1].t if idx + 1 < len(designs) else None
    design_visits = [
        v for v in visits
        if v.t >= t_start and (t_end is None or v.t < t_end)
    ]
    visit_ids = sorted({v.visit_id for v in design_visits})
    n = len(visit_ids)
    visit_str = ', '.join(str(v) for v in visit_ids[:30])
    if n > 30:
        visit_str += f' … ({n} total)'
    did_hex = hex(dr.design_id)
    design_info_md.object = (
        f'**Design:** `{did_hex}`  \n'
        f'**{n} visit{"s" if n != 1 else ""}:** {visit_str}'
    )


@pn.depends(design_select.param.value, watch=True)
def _on_design(value):
    _update_design_info(value)
    if not _busy[0]:
        _render()


@pn.depends(panels_check.param.value, watch=True)
def _on_panels(value):
    _render()


@pn.depends(show_acq_frames.param.value, watch=True)
def _on_show_acq(value):
    _render()


@pn.depends(full_offset_range.param.value, watch=True)
def _on_full_offset_range(value):
    _render()


@pn.depends(guide_coords.param.value, watch=True)
def _on_guide_coords(value):
    _render()


@pn.depends(tail_window_select.param.value, watch=True)
def _on_tail_window(value):
    if follow_toggle.value:
        _render()


@pn.depends(follow_toggle.param.value, watch=True)
def _on_follow(active):
    _stop_follow()
    if active:
        follow_toggle.name = '⏸ Following…'
        _init_tail()
        _tail_tick()
        _follow_cb[0] = pn.state.add_periodic_callback(
            _tail_tick, period=interval_input.value * 1000,
        )
    else:
        follow_toggle.name = '▶ Follow / Tail'
        _render()


@pn.depends(show_all_files.param.value, watch=True)
def _on_show_all(value):
    _stop_follow()
    follow_toggle.value = False
    _busy[0] = True
    _refresh_file_list(dir_input.value)
    _busy[0] = False
    _load_file()


# ── Layout ────────────────────────────────────────────────────────────────────

_HELP_TEXT = """
## AG Explorer — Panel Guide

| Panel | What it shows | What to look for |
|-------|--------------|-----------------|
| **Guide Offsets** | Per-frame RA, Dec, Focus, and Scale corrections sent to the telescope (arcsec / mm). Grey markers = INVALID_OFFSET (computed but not applied). | Spikes in dRA or dDec indicate poor star matches or a seeing event. Persistent nonzero offset suggests a pointing drift. Correlate spikes with the Camera Counts heatmap to see if a detector dropout is the cause. |
| **InR & Scale** | Instrument-rotator correction (dInR, arcsec) and differential scale (dScale, ×10⁻⁶). Default y-range ±10 arcsec / ±100×10⁻⁶; unlock with "full y-range". | Large dInR spikes may indicate a rotator calibration issue or a bad frame match. dScale drifting away from zero can signal a plate-scale change (focus related). |
| **Focus** | Per-camera focus offset (mm) from glass-plate differential focus sensing: the RIGHT half of each detector has a plain glass plate; the PSF second moments differ between glass and no-glass halves in proportion to the focus error. | Divergence between cameras suggests field tilt. A systematic nonzero offset across all cameras means M2 needs adjustment. |
| **Camera-half detections (heatmap)** | Each row = one detector half (AG1L–AG6R). Color = (raw detected sources on that half) / (median count for that half across all guiding frames in the same PFS design/pointing). *Count* is every spot found by the detection algorithm on that half — before any catalog matching or quality filtering. *Design median* is the per-half baseline for the current field, so the ratio automatically adapts to star density. White = normal (ratio≈1), red = dropout (ratio≈0, sources vanished), blue = elevated (ratio≈2+). Only frames where guiding was actively running appear. | A sudden red cell on one half — especially coinciding with a guide-offset spike — is the detector-half dropout event. A red half while its partner stays white (e.g. AG3R drops, AG3L normal) confirms a detector-half issue rather than a whole-camera or field problem. |
| **Detected counts per camera-half** | Raw per-half detection counts over time as line traces (12 traces). | Use alongside the heatmap to see absolute magnitudes. A count near zero on one trace while others are normal = half-detector dropout. |
| **Star quality / seeing proxy** | Median flux (ADU), peak ADU, and PSF size (arcsec, seeing proxy) across **all matched guide stars** in the frame — not the brightest star. The median is taken with `nanmedian` so a single saturated or near-saturated star cannot spike the value. | Rising PSF size = seeing degrading. If PSF size spikes at the same frame as a guide offset spike, the seeing caused the problem rather than a detector issue. If peak ADU spikes while PSF size is normal, check for a near-saturated star entering the match. |
| **Telescope** | Azimuth, elevation, parallactic angle (rot, slow ~0.04°/min) and mechanical rotator position (InR, jumps between visits) from Gen2. | `rot` changes smoothly as the sky tracks; `InR` jumps at the start of each visit. Large InR values at visit boundaries are expected. |

**Scatter plots (right column)**
- *dAz vs dEl*: guide offset cloud; tight cluster = stable guiding.
- *dInR vs dScale*: outliers in dInR indicate rotator instability.
- *dFocus vs counts*: checks whether focus correlates with detection counts.
- *Peak ADU vs PSF size*: saturation / seeing correlation. Peak ADU here is the **median** peak intensity across matched stars (not the brightest star), so sustained elevation means the typical matched star is near saturation.

**Colour coding** — mode: 🟢 autoguide · 🟠 acquire · 🟣 converge. Grey × markers = INVALID_OFFSET frames.
"""

_DATA_SOURCES_TEXT = """## AG Explorer — Data Sources & Computation

Each column is tagged **Measured** (raw hardware/sensor value), **Computed** (derived from other values), or **Aggregated** (statistic over a set of measured values).

---

### Guide Corrections
*All guide correction values come from the astrometric least-squares solve run on each AG exposure.*

| Quantity | Source | Type | Details |
|----------|--------|------|---------|
| dRA, dDec | **DB** `agc_guide_offset.ra_offset / dec_offset`; **Log** `guideErrors` fields 2–3 | Computed | Least-squares solution of matched-star residuals decomposed into RA/Dec axes (arcsec). |
| dInR | **DB** `agc_guide_offset.inr_offset`; **Log** `guideErrors` field 4 | Computed | Rotational component of the astrometric residual (arcsec). |
| dAz, dEl | **DB** `agc_guide_offset.az_offset / el_offset`; **Log** `guideErrors` fields 5–6 | Computed | Guide correction projected onto azimuth/elevation axes (arcsec). |
| dFocus | **DB** `agc_guide_offset.focus_offset`; **Log** `guideErrors` field 7 | Computed | Telescope-level focus correction (mm); this is the single aggregate value sent to M2, not the per-camera focus below. |
| dScale | **DB** `agc_guide_offset.scale_offset`; **Log** `guideErrors` field 8 | Computed | Differential scale factor from the astrometric fit (×10⁻⁶). |
| Status | **Log** `guideErrors` field 9 | Computed | `OK` = correction applied; `ERROR` = solve failed; `INVALID_OFFSET` = correction computed but not sent (exceeded `max_correction` threshold). |

---

### Star Quality (Star Stats panel)
*All three quantities are computed per frame from the set of **matched** guide stars only (catalog-matched detections passing bad-flag filtering), then summarised with `nanmedian`.*

| Quantity | Source | Type | Details |
|----------|--------|------|---------|
| Flux | **DB** `agc_detected.image_moment_00_pix` | Measured → Aggregated | Zeroth image moment = total pixel sum inside the detection footprint (ADU). `nanmedian` across all matched guide stars in the frame. |
| Peak ADU | **DB** `agc_detected.peak_intensity` | Measured → Aggregated | Brightest single pixel inside the detection footprint (ADU). `nanmedian` across all matched guide stars — **not** the brightest star in the field; one saturated source cannot spike this value. |
| PSF size | **DB** `agc_detected.central_image_moment_20_pix`, `_11_pix`, `_02_pix` | Computed → Aggregated | Effective seeing-disc diameter in arcsec. Formula: `plate_scale × 2 × √(a·b)`, where *a*, *b* are the square-root eigenvalues of the 2×2 second-moment matrix `[[m20, m11], [m11, m02]]`. Plate scale = 206.265 × 13 µm/px ÷ 15 000 mm ≈ 0.179 arcsec/px. `nanmedian` across matched stars. |

---

### Camera-Half Detections (heatmap & count traces)

| Quantity | Source | Type | Details |
|----------|--------|------|---------|
| Count | **DB** `agc_detected`, `COUNT(*)` grouped by `(agc_camera_id, flags & 1)` | Measured | Raw detected sources on each half **before** any catalog matching or quality filtering. `flags & 1` = 0 → left half, 1 → right half. Camera IDs are 0-based (0–5) in DB; displayed as AG1–AG6. |
| Design median | Computed from counts above | Computed | Median of per-half counts over all guiding frames within the same PFS design/pointing. Used to normalise heatmap colour so star-density differences between fields cancel out. |

---

### Per-Camera Focus (Focus panel)

| Quantity | Source | Type | Details |
|----------|--------|------|---------|
| z1–z6 | **Log** `focus` reply line from agActor | Computed | Per-camera focus offset (mm) from glass-plate differential focus sensing. Each AG detector has a plain glass plate on its RIGHT half (`SourceDetectionFlag.RIGHT`, `flag==1`) and no glass on the LEFT half (`flag==0`). The glass shifts the effective focal plane, making PSF second moments differ between halves. Formula: `focus_error [mm] = [median(a²+b²)_no_glass − median(a²+b²)_with_glass] × 0.0086 − 0.026`. Not stored per-camera in DB; extracted from log `FocusRecord`. Divergence between cameras indicates field tilt. |

---

### Telescope State (Telescope panel)

| Quantity | Source | Type | Details |
|----------|--------|------|---------|
| az, el | **Log** Gen2 `tel_axes` reply (~10 s cadence) | Measured | Telescope azimuth and elevation (degrees) from Gen2 encoder readout. |
| rot | **Log** Gen2 `tel_axes` reply, 3rd field (~10 s cadence) | Measured | **Parallactic / field rotation angle** (°). Changes smoothly at ~0.04°/min as the field tracks. Typical range 20–55°. **Not** the mechanical instrument rotator position. |
| InR (rotator) | **DB** `agc_exposure_info.insrot`; **Log** `tel_rot` reply (~1 s cadence) | Measured | Instrument rotator mechanical position (°, range ±180°). Jumps between visits as the rotator slews to the required PA. Differs from `rot` by tens of degrees — do not treat them as equivalent. |
| adc, m2_pos3 | **Log** `taken_at=…` line (acquire-phase only) | Measured | ADC prism angle (°) and M2 secondary mirror Z-position (mm). Emitted only during `acquire_field`, not during regular autoguide frames. |
"""

help_modal = pn.layout.Modal(
    pn.Column(
        pn.Tabs(
            ('Panel Guide',   pn.pane.Markdown(_HELP_TEXT, width=900)),
            ('Data Sources',  pn.pane.Markdown(_DATA_SOURCES_TEXT, width=900)),
        ),
        styles={
            'overflow-y': 'auto',
            'max-height': '72vh',
            'overscroll-behavior': 'contain',
        },
        width=940,
    ),
    name='AG Explorer Help',
)

help_btn = pn.widgets.Button(name='? Help', button_type='light', width=280, margin=(2, 5))

def _on_help(event):
    help_modal.open = True

help_btn.on_click(_on_help)


# ── Per-panel maximize (sidebar select) ──────────────────────────────────────
_MAXIMIZE_PLACEHOLDER = '— select panel —'

_maximize_pane = pn.pane.Plotly(go.Figure(), sizing_mode='stretch_width', height=700)

_maximize_modal = pn.layout.Modal(
    pn.Column(
        _maximize_pane,
        sizing_mode='stretch_width',
        styles={'overflow': 'hidden', 'min-width': '85vw'},
    ),
    name='Panel Detail',
)

maximize_select = pn.widgets.Select(
    name='Maximize panel',
    options=[_MAXIMIZE_PLACEHOLDER] + PANEL_NAMES + ['Scatter plots'],
    value=_MAXIMIZE_PLACEHOLDER,
    width=270,
)


def _do_maximize(panel_name):
    store = _current_store[0]
    if store is None:
        return
    if panel_name == 'Scatter plots':
        fig = _build_scatter_column(
            store, show_acq_frames.value, clamp_offsets=not full_offset_range.value,
        )
        fig.update_layout(width=None, height=900)
    else:
        x_start, x_end = _current_x_range
        if x_start is None:
            return
        fig = _build_figure(store, [panel_name], x_start, x_end, db_data=_current_db_data[0])
        fig.update_layout(height=700, margin=dict(l=80, r=20, t=40, b=50))
    _maximize_pane.object = fig
    _maximize_modal.open = True


@pn.depends(maximize_select.param.value, watch=True)
def _on_maximize_select(value):
    if value == _MAXIMIZE_PLACEHOLDER:
        return
    _do_maximize(value)
    maximize_select.value = _MAXIMIZE_PLACEHOLDER

sidebar = pn.Column(
    pn.pane.Markdown('### Controls', margin=(5, 5, 0, 5)),
    help_btn,
    dir_input,
    file_select,
    show_all_files,
    fetch_db_btn,
    design_select,
    pn.layout.Divider(),
    pn.pane.Markdown('**Panels**', margin=(0, 5)),
    panels_check,
    maximize_select,
    show_acq_frames,
    full_offset_range,
    guide_coords,
    pn.layout.Divider(),
    pn.Row(follow_toggle, interval_input),
    tail_window_select,
    pn.layout.Divider(),
    status_md,
    db_status_md,
    design_info_md,
    width=310,
    sizing_mode='stretch_height',
)

app = pn.Column(
    help_modal,
    _maximize_modal,
    pn.Row(
        sidebar,
        pn.Column(plot_pane, cam_counts_pane, sizing_mode='stretch_both'),
        scatter_col_pane,
        sizing_mode='stretch_both',
    ),
    sizing_mode='stretch_both',
)

app.servable()

# Initial population (runs in notebook; panel serve calls servable() instead)
_refresh_file_list(dir_input.value)
